In [ ]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
import myfunction as mf
import myuncertain as mu
import os
from path_config import *

path_5_sw = drive_letter + "/wyy/code_project/running_outcome/final_data/SPDB/part3_forecast/sw_forecast/sw_lgbm_output/only_pfas/"
path_5_lr = drive_letter + "/wyy/code_project/running_outcome/final_data/SPDB/part3_forecast/lr_forecast/lr_lgbm_output/only_pfas/"
input_dir = r'E:\wyy\code_project\running_outcome\final_data\SPDB\part3_forecast\sw_forecast\sw_lgbm_output\only_year'

list_pfas =['PFOA', 'PFNA', 'PFDA', 'PFUnDA','PFDoDA','PFTrDA', 'PFTeDA', 'PFHxS', 'PFOS', 'FOSA', 'PFBA', 'PFPeA', 'PFHxA', 'PFHpA','PFBS','HFPO-DA']


### 参数准备

In [ ]:

df_country = pd.read_excel(path_file_csv + "hi_raw.xlsx",sheet_name='country')
dic_code = df_country.set_index("country_ename")["country_id"].to_dict()

df_weight = pd.read_excel(path_file_csv + "hi_raw.xlsx",sheet_name='bmi')
df_weight["country_id"] = df_weight["country_ename"].map(dic_code)
df_weight = df_weight[['country_id','weight']]


weight_mean = df_weight['weight'].mean()


all_country_ids = df_country['country_id'].unique()


existing_country_ids = df_weight['country_id'].unique()


missing_country_ids = set(all_country_ids) - set(existing_country_ids)


missing_data = pd.DataFrame({'country_id': list(missing_country_ids), 'weight': weight_mean})
df_weight = pd.concat([df_weight, missing_data])
df_weight = df_weight[(df_weight['country_id'].notna())&(df_weight['country_id']!=0)]

df_weight.to_csv(path_part4_hi + 'weight.csv',index=False)

In [ ]:

df_life = pd.read_excel(path_file_csv + "hi_raw.xlsx",sheet_name='life')
df_life = df_life[(df_life['Year']>1999)&(df_life['Year']<2021)]

df_avg_life = df_life[['country_ename', "life_ex", 'Year']]
df_avg_life = df_avg_life.rename(columns={'Year':'year'})
df_country = pd.read_excel(path_file_csv + "hi_raw.xlsx",sheet_name='country')
dic_code = df_country.set_index("country_ename")["country_id"].to_dict()

df_avg_life["country_id"] = df_avg_life["country_ename"].map(dic_code)
df_avg_life = df_avg_life[['country_id','life_ex', 'year']]


all_country_ids = df_country['country_id'].unique()


existing_country_ids = df_avg_life['country_id'].unique()


missing_country_ids = set(all_country_ids) - set(existing_country_ids)


all_years = df_avg_life['year'].unique()


yearly_life_mean = df_avg_life.groupby('year')['life_ex'].mean()


missing_data = []
for country_id in missing_country_ids:
    for year in all_years:
        missing_data.append({
            'country_id': country_id, 
            'life_ex': yearly_life_mean[year], 
            'year': year
        })

missing_df = pd.DataFrame(missing_data)


df_avg_life = pd.concat([df_avg_life, missing_df])

df_avg_life = df_avg_life[(df_avg_life['country_id'].notna())&(df_avg_life['country_id']!=0)]
df_avg_life.to_csv(path_part4_hi + 'life_ex.csv',index=False)

In [ ]:

df_country = pd.read_excel(path_file_csv + "hi_raw.xlsx", sheet_name='country')
dic_code = df_country.set_index("country_ename")["country_id"].to_dict()

df_consume = pd.read_excel(path_file_csv + "hi_raw.xlsx", sheet_name='consume')
df_consume = df_consume[(df_consume['Year Code']>1999)&(df_consume['Year Code']<2021)].copy()
df_consume = df_consume.rename(columns={'Year Code':'year'})


df_consume_sf = df_consume[df_consume['Item']=='Marine Fish, Other'].copy()
df_consume_ff = df_consume[df_consume['Item']=='Freshwater Fish'].copy()


df_consume_sf.loc[:, "country_id"] = df_consume_sf["country_ename"].map(dic_code)
df_consume_sf_avg = df_consume_sf[['country_id','consume','year']].copy()


yearly_sf_mean = df_consume_sf_avg.groupby('year')['consume'].mean()

all_country_ids = df_country['country_id'].unique()
all_years = df_consume_sf_avg['year'].unique()


all_combinations = pd.DataFrame([(country, year) for country in all_country_ids for year in all_years], 
                               columns=['country_id', 'year'])


all_combinations['country_id'] = all_combinations['country_id'].astype(df_consume_sf_avg['country_id'].dtype)
all_combinations['year'] = all_combinations['year'].astype(df_consume_sf_avg['year'].dtype)

df_consume_sf_avg = pd.merge(all_combinations, df_consume_sf_avg, 
                             on=['country_id', 'year'], how='left')

df_consume_sf_avg.to_csv(path_part4_hi + 'sf_consume.csv', index=False)


df_consume_ff.loc[:, "country_id"] = df_consume_ff["country_ename"].map(dic_code)
df_consume_ff_avg = df_consume_ff[['country_id','consume','year']].copy()


yearly_ff_mean = df_consume_ff_avg.groupby('year')['consume'].mean()


all_combinations['country_id'] = all_combinations['country_id'].astype(df_consume_ff_avg['country_id'].dtype)
all_combinations['year'] = all_combinations['year'].astype(df_consume_ff_avg['year'].dtype)


df_consume_ff_avg = pd.merge(all_combinations, df_consume_ff_avg, 
                            on=['country_id', 'year'], how='left')

df_consume_ff_avg.to_csv(path_part4_hi + 'ff_consume.csv', index=False)


In [ ]:


df = pd.read_csv(path_part4_hi + 'rfd_raw.csv')
df = df.sort_values(['PFAS', 'orz', 'year'], ascending=[True, True, False])
df = df.drop_duplicates(['PFAS', 'orz'], keep='first')

result = df.groupby('PFAS')['RfD'].agg(['min', 'max', 'median']).reset_index()

result.columns = ['PFAS', 'min', 'max', 'median']


print(result.to_string(index=False))
result.to_csv(path_part4_hi + 'rfd.csv', index=False)


   PFAS     min    max  median
   FOSA   12.00   12.0   12.00
HFPO-DA    3.00   77.0   40.00
   PFBA 2900.00 3000.0 2950.00
   PFBS  230.00 1400.0  365.00
   PFDA    5.00   12.0    8.50
 PFDoDA   12.00   12.0   12.00
  PFHpA    5.00   23.0   14.00
  PFHxA  500.00  500.0  500.00
  PFHxS    3.80   20.0    7.35
   PFNA    0.74   12.0    3.65
   PFOA    0.03   18.0    4.00
   PFOS    0.10   23.0    2.50
  PFPeA  500.00  500.0  500.00
 PFTeDA   12.00   12.0   12.00
 PFTrDA   12.00   12.0   12.00
 PFUnDA   12.00   12.0   12.00


### 网格化

In [ ]:
df_country_id = pd.read_csv(path_part4_grid + 'country_id.csv')
df_country_id = df_country_id[df_country_id['country_id']!=0]

df_population = pd.read_csv(path_part4_grid + 'population.csv')

df_coast = pd.read_csv(path_part4_grid + 'coast_distance.csv')

df_type = pd.read_csv(path_part4_grid + 'type.csv')

df_merged = pd.merge(df_country_id, df_population, on=['lon', 'lat'], how='left')
df_merged = pd.merge(df_merged, df_type, on=['lon', 'lat'], how='left')

df_merged = pd.merge(df_merged, df_coast, on=['lon', 'lat'], how='left')


population_nan = df_merged['pop'].isna().sum()
print(f"Number of unmatched population data: {population_nan}")

type_nan = df_merged['type'].isna().sum()
print(f"Number of unmatched type data: {type_nan}")
type_nan = df_merged['coast_distance'].isna().sum()
print(f"Number of unmatched type data: {type_nan}")
print(df_merged.columns)
df_merged.to_csv(path_part4_grid + 'grid.csv',index=False)

Number of unmatched population data: 0
Number of unmatched type data: 0
Number of unmatched type data: 0
Index(['lat', 'lon', 'country_id', 'pop', 'type', 'coast_distance'], dtype='object')


In [ ]:


csv_files = [f for f in os.listdir(input_dir) if f.endswith('.csv')]


all_data = pd.DataFrame()
list_remain_sw = ['lon_grid','lat_grid', 'year'] + list_pfas


for csv_file in csv_files:

    df = pd.read_csv(os.path.join(input_dir, csv_file))
    str_year = csv_file[3:7]
    df['year'] = str_year
    df = df.drop(['value', 'lc_value', 'sc_value'], axis=1)


    all_data = pd.concat([all_data, df], ignore_index=True)


all_data_remain = all_data[list_remain_sw]
all_data_remain.to_csv(os.path.join(path_part4_pfas, 'sw_concentration.csv'), index=False)


In [ ]:
df_type = pd.read_csv(path_part4_grid + 'type.csv')
df_sw = pd.read_csv(path_part4_pfas + 'sw_concentration.csv')
df_sw = df_sw.rename(columns={'lon_grid':'lon','lat_grid':'lat'})
df_sw =df_sw[df_sw['year']==2020]

df_grid = pd.read_csv(path_part4_grid + 'grid.csv')

df_sw_type = pd.merge(df_sw, df_type, on=['lon', 'lat'], how='left')
df_sw_type = df_sw_type.rename(columns={'type':'forecast_type'})
df_sw_type = df_sw_type[['lon','lat','forecast_type']]
df_grid_final = pd.merge(df_grid, df_sw_type, on=['lon', 'lat'], how='left')

df_grid_final.to_csv(path_part4_grid + 'new_grid.csv',index=False)

### 最近网格

In [ ]:
import pandas as pd
import numpy as np


def haversine(lat1, lon1, lat2, lon2):
    """
    Calculate the great circle distance in kilometers between two points 
    on the earth (specified in decimal degrees)
    """

    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])

    dlon = lon2 - lon1 
    dlat = lat2 - lat1 
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a)) 
    r = 6371 # 地球平均半径，单位为公里
    return c * r


def get_nearest_grid(df_need, df_search, mode='c'):
    nearest_grid_lon = []
    nearest_grid_lat = []

    for idx, row in df_need.iterrows():
        lat, lon, country_id = row['lat'], row['lon'], row['country_id']


        df_search['distance'] = haversine(lat, lon, df_search['lat'], df_search['lon'])

        if mode == 'c':  # 按照国家匹配的逻辑
            grid1_same_country = df_search[df_search['country_id'] == country_id].nsmallest(1, 'distance')
            if grid1_same_country.empty:
                grid1_diff_country = df_search[df_search['country_id'] != country_id].nsmallest(1, 'distance')
                nearest_grid_lon.append(grid1_diff_country.iloc[0]['lon'])
                nearest_grid_lat.append(grid1_diff_country.iloc[0]['lat'])
            else:
                nearest_grid_lon.append(grid1_same_country.iloc[0]['lon'])
                nearest_grid_lat.append(grid1_same_country.iloc[0]['lat'])

        elif mode == 'd':  # 不考虑国家，直接找最近的网格
            nearest_grid = df_search.nsmallest(1, 'distance')
            nearest_grid_lon.append(nearest_grid.iloc[0]['lon'])
            nearest_grid_lat.append(nearest_grid.iloc[0]['lat'])

    return nearest_grid_lon, nearest_grid_lat


In [4]:
df_grid = pd.read_csv(path_parts_data + 'grid_re.csv')
df_grid_need = df_grid[(df_grid['pop_type'].notna()) & (df_grid['pop_type']==1)]
df_need1 = df_grid_need.copy()
print(df_need1.shape)
df_need1 = df_need1[['lon','lat','country_id']]

df_search1 = df_grid[(df_grid['ocean_water']==1)&(df_grid['ocean_water'].notna())]
print(df_search1.shape)
df_search1 = df_search1[['lon','lat','country_id']]

df_need2 = df_grid_need.copy()
print(df_need2.shape)
df_need2 = df_need2[['lon','lat','country_id']]

df_search2 = df_grid[(df_grid['inland_water']==0)&(df_grid['inland_water'].notna())]
print(df_search2.shape)
df_search2 = df_search2[['lon','lat','country_id']]

(17872, 8)
(42695, 8)
(17872, 8)
(53809, 8)


In [ ]:
sf_nearest_grid_lon, sf_nearest_grid_lat = get_nearest_grid(df_need1, df_search1, 'd')
df_need1['sf_nearest_lon'] = sf_nearest_grid_lon
df_need1['sf_nearest_lat'] = sf_nearest_grid_lat

df_need1 = df_need1[['lon','lat','sf_nearest_lon', 'sf_nearest_lat']]


ff_nearest_grid_lon, ff_nearest_grid_lat = get_nearest_grid(df_need2, df_search2, 'd')
df_need2['ff_nearest_lon'] = ff_nearest_grid_lon
df_need2['ff_nearest_lat'] = ff_nearest_grid_lat

df_need2 = df_need2[['lon','lat','ff_nearest_lon', 'ff_nearest_lat']]



df_neareat = pd.merge(df_need1, df_need2, on=['lon', 'lat'], how='left')
df_neareat.to_csv(path_part4_grid + 'nearest_grid_d.csv',index=False)

### 网格消费量

In [ ]:
df_grid_re = pd.read_csv(path_parts_data + 'grid_re.csv')
df_pop_year = pd.read_csv(path_part4_grid + 'population_year.csv')

merged_df = pd.merge(df_pop_year, df_grid_re, on=['lon', 'lat'], how='left')
merged_df.to_csv(path_part4_grid + 'new_grid_year.csv',index=False)

pop_summary = merged_df.groupby(['country_id', 'year'])['pop'].sum().reset_index()


pop_summary.to_csv(path_part4_hi + 'country_pop.csv',index=False)


In [ ]:
import pandas as pd
import numpy as np

df_country = pd.read_csv(path_part4_hi + 'country_pop.csv')
df_country = df_country[df_country['pop'] > 0]

df_ff_consume = pd.read_csv(path_part4_hi + 'ff_consume.csv')
df_ff_consume['consume'] = df_ff_consume['consume'] / 365
df_ff_consume = df_ff_consume.rename(columns={'consume': 'ff_consume'})

df_sf_consume = pd.read_csv(path_part4_hi + 'sf_consume.csv')
df_sf_consume['consume'] = df_sf_consume['consume'] / 365
df_sf_consume = df_sf_consume.rename(columns={'consume': 'sf_consume'})


df_country = pd.merge(df_country, df_ff_consume[['country_id', 'year', 'ff_consume']], 
                      on=['country_id', 'year'], how='left')
df_country = pd.merge(df_country, df_sf_consume[['country_id', 'year', 'sf_consume']], 
                      on=['country_id', 'year'], how='left')


def fill_missing(df, column):

    df[column] = df.groupby('country_id')[column].transform(lambda x: x.fillna(x.median()))


    global_median = df.groupby('year')[column].transform('median')
    df[column] = df[column].fillna(global_median)

    return df


df_country = fill_missing(df_country, 'ff_consume')
df_country = fill_missing(df_country, 'sf_consume')


df_country['ff_consume_sum'] = df_country["ff_consume"] * df_country["pop"]
df_country['sf_consume_sum'] = df_country["sf_consume"] * df_country["pop"]


df_country.to_csv(path_part4_hi + 'avg_consume_median.csv', index=False)

In [ ]:
import pandas as pd
import numpy as np

def process_year_data(df_grid, df_avg_consume, year):
    df_grid_year = df_grid[df_grid['year'] == year]
    df_avg_consume_year = df_avg_consume[df_avg_consume['year'] == year]

    df_grid_inland = df_grid_year[df_grid_year['pop'].notna()]

    dict_sf = df_avg_consume_year.set_index("country_id")["sf_consume"].to_dict()
    dict_ff = df_avg_consume_year.set_index("country_id")["ff_consume"].to_dict()

    df_grid_inland['sf_consume'] = df_grid_inland["country_id"].map(dict_sf)
    df_grid_inland['ff_consume'] = df_grid_inland["country_id"].map(dict_ff)

    df_grid_inland['ad_fa1_num'] = 0.22222 * np.exp(-0.002 * df_grid_inland['coast_distance'])

    max_val = df_grid_inland['ad_fa1_num'].max()
    min_val = df_grid_inland['ad_fa1_num'].min()
    list_id = df_grid_inland['country_id'].unique().tolist()

    df_grid_inland['ad_fa_per_sf'] = (df_grid_inland['ad_fa1_num'] - min_val) / (max_val - min_val)

    for country_id in list_id:
        type_0_count = df_grid_inland[df_grid_inland['country_id'] == country_id].shape[0]

        if type_0_count == 1:
            df_grid_inland.loc[(df_grid_inland['country_id'] == country_id), 'ad_fa_per_ff'] = 1
        else:
            df_grid_inland.loc[(df_grid_inland['country_id'] == country_id), 'ad_fa_per_ff'] = 1 - df_grid_inland.loc[(df_grid_inland['country_id'] == country_id), 'ad_fa_per_sf']

    df_grid_inland['re1_sf_consume'] = df_grid_inland['sf_consume'] * df_grid_inland['ad_fa_per_sf']
    df_grid_inland['re1_ff_consume'] = df_grid_inland['ff_consume'] * df_grid_inland['ad_fa_per_ff']

    df_grid_inland['re1_sf_consume_sum'] = df_grid_inland['re1_sf_consume'] * df_grid_inland['pop']
    df_grid_inland['re1_ff_consume_sum'] = df_grid_inland['re1_ff_consume'] * df_grid_inland['pop']

    sf_consume_sum_dict = df_grid_inland.groupby('country_id')['re1_sf_consume_sum'].sum().to_dict()
    ff_consume_sum_dict = df_grid_inland.groupby('country_id')['re1_ff_consume_sum'].sum().to_dict()

    df_avg_consume_year.loc[:, 're1_sf_consume_sum'] = df_avg_consume_year['country_id'].map(sf_consume_sum_dict)
    df_avg_consume_year.loc[:, 're1_ff_consume_sum'] = df_avg_consume_year['country_id'].map(ff_consume_sum_dict)

    df_avg_consume_year['tf_ff'] = df_avg_consume_year['ff_consume_sum'] / df_avg_consume_year['re1_ff_consume_sum']
    df_avg_consume_year['tf_ff'].fillna(1, inplace=True)

    df_avg_consume_year['tf_sf'] = df_avg_consume_year['sf_consume_sum'] / df_avg_consume_year['re1_sf_consume_sum']
    df_avg_consume_year['tf_sf'].fillna(1, inplace=True)

    dict_sf_fa = df_avg_consume_year.set_index("country_id")["tf_sf"].to_dict()
    dict_ff_fa = df_avg_consume_year.set_index("country_id")["tf_ff"].to_dict()

    df_grid_inland['tf_sf'] = df_grid_inland['country_id'].map(dict_sf_fa)
    df_grid_inland['tf_ff'] = df_grid_inland['country_id'].map(dict_ff_fa)

    df_grid_inland['re2_sf_consume'] = df_grid_inland['re1_sf_consume'] * df_grid_inland['tf_sf']
    df_grid_inland['re2_ff_consume'] = df_grid_inland['re1_ff_consume'] * df_grid_inland['tf_ff']

    df_grid_inland_output = df_grid_inland[['lon','lat','country_id', 're2_sf_consume','re2_ff_consume','pop','coast_distance', 'year']]
    df_grid_inland_output = df_grid_inland_output.rename(columns={'re2_sf_consume':'sf_consume','re2_ff_consume':'ff_consume'})
    df_grid_inland_output = df_grid_inland_output[df_grid_inland_output['pop']>0]

    return df_grid_inland_output


df_grid = pd.read_csv(path_part4_grid + 'new_grid_year.csv')

df_avg_consume = pd.read_csv(path_part4_hi + 'avg_consume_median.csv')


years = df_grid['year'].unique()


results = []
for year in years:
    result = process_year_data(df_grid, df_avg_consume, year)
    results.append(result)


final_result = pd.concat(results, ignore_index=True)


final_result.to_csv(path_part4_grid + 'consume_grid_year.csv', index=False)

### 匹配网格PFAS 

In [11]:
df_po = pd.read_excel(path_file + inf_file, sheet_name="po_treat")
list_pfas_all = df_po['posname'].tolist()
list_pfas_all_lc = df_po[df_po['po_chain']==1]['posname'].tolist()
list_pfas_all_sc = df_po[df_po['po_chain']==0]['posname'].tolist()
print(len(list_pfas_all_lc))
print(len(list_pfas_all_sc))
list_list_pfas = [list_pfas_all, list_pfas_all_lc, list_pfas_all_sc]

convert = mu.create_converter(df_po)


37
23


In [ ]:
import pandas as pd
df_grid = pd.read_csv(path_part4_grid + 'consume_grid_year.csv')

df_nearest = pd.read_csv(path_part4_grid + 'nearest_grid_d.csv')

df_grid = df_grid.merge(df_nearest, on=['lon', 'lat'], how='left')


for str_pfas in list_pfas:
   safe_pfas_name = convert(str_pfas)
   df_lr = pd.read_csv(path_5_lr + 'lr_'+safe_pfas_name+'.csv')
   df_sw = pd.read_csv(path_5_sw + 'sw_'+safe_pfas_name+'.csv')

   for df_select in [df_lr, df_sw]:
      if df_select is df_lr:
         str_marke = 'lr'
         df_data = df_lr.copy()
      else:
         str_marke = 'sw'
         df_data = df_sw.copy()
      print(str_marke)
      df_grid_pfas = df_grid.copy()
      print(df_grid_pfas.columns)
      list_remain = ['lon','lat','year','country_id','ff_consume','sf_consume',str_pfas + '_min',str_pfas + '_max',str_pfas + '_median']
      df_grid_pfas['sw_consume'] = 2*0.2*0.5 # 日常建议水摄入量2L 20%来自饮用水 50%的饮用水来自地表水

      df_grid_pfas = df_grid_pfas.merge(df_data, left_on=['ff_nearest_lon', 'ff_nearest_lat', 'year'], right_on=['lon_grid', 'lat_grid', 'year'], how='left')
      df_grid_pfas = df_grid_pfas[list_remain]
      df_grid_pfas.to_csv(path_part4_pfas + str_marke + '_' + safe_pfas_name+'.csv', index=False)


lr
Index(['lon', 'lat', 'country_id', 'sf_consume', 'ff_consume', 'pop',
       'coast_distance', 'year', 'sf_nearest_lon', 'sf_nearest_lat',
       'ff_nearest_lon', 'ff_nearest_lat'],
      dtype='object')
sw
Index(['lon', 'lat', 'country_id', 'sf_consume', 'ff_consume', 'pop',
       'coast_distance', 'year', 'sf_nearest_lon', 'sf_nearest_lat',
       'ff_nearest_lon', 'ff_nearest_lat'],
      dtype='object')
lr
Index(['lon', 'lat', 'country_id', 'sf_consume', 'ff_consume', 'pop',
       'coast_distance', 'year', 'sf_nearest_lon', 'sf_nearest_lat',
       'ff_nearest_lon', 'ff_nearest_lat'],
      dtype='object')
sw
Index(['lon', 'lat', 'country_id', 'sf_consume', 'ff_consume', 'pop',
       'coast_distance', 'year', 'sf_nearest_lon', 'sf_nearest_lat',
       'ff_nearest_lon', 'ff_nearest_lat'],
      dtype='object')
lr
Index(['lon', 'lat', 'country_id', 'sf_consume', 'ff_consume', 'pop',
       'coast_distance', 'year', 'sf_nearest_lon', 'sf_nearest_lat',
       'ff_nearest_lon', '

### 网格人口

In [ ]:

country_df = pd.read_csv(path_part4_hi + 'country.csv')
wiw_df = pd.read_csv(path_part4_hi + 'without_improved_water.csv')
pop_df = pd.read_csv(path_part4_hi + 'country_pop.csv')


country_map = country_df[['country_id', 'country_code', 'region']].set_index('country_code')


wiw_df = wiw_df.merge(country_map, left_on='country_code', right_index=True)


wiw_df = wiw_df[['country_id', 'year', 'wiw', 'region']]


wiw_df['wiw'] = wiw_df.groupby('country_id')['wiw'].transform(lambda x: x.fillna(x.median()))
wiw_df['wiw'] = wiw_df['wiw'].fillna(wiw_df.groupby('year')['wiw'].transform('median'))


wiw_df['wiw'] = wiw_df['wiw'].round().astype(int)

result_df = wiw_df.merge(pop_df, on=['country_id', 'year'])


result_df['per_wiw'] = (result_df['wiw'] / result_df['pop']).round(8)


def adjust_per_wiw(row, median_per_wiw):
    if row['pop'] == 0 or row['wiw'] == 0:
        return median_per_wiw
    return row['per_wiw']


for (region, year), group in result_df.groupby(['region', 'year']):
    median_per_wiw = group['per_wiw'].median()
    result_df.loc[(result_df['region'] == region) & (result_df['year'] == year), 'per_wiw'] = \
        result_df.loc[(result_df['region'] == region) & (result_df['year'] == year)].apply(
            adjust_per_wiw, axis=1, median_per_wiw=median_per_wiw)

output_file_per_wiw = os.path.join(path_part4_hi, "per_wiw.csv")

result_df.to_csv(output_file_per_wiw, index=False)

print("处理完成，结果已保存到per_wiw.csv")

处理完成，结果已保存到per_wiw.csv


In [14]:
df_per_wiw = pd.read_csv(output_file_per_wiw)
df_per_wiw = df_per_wiw[['year','country_id','per_wiw']]
df_grid = pd.read_csv(path_part4_grid + 'new_grid_year.csv')
df_grid = df_grid[['lon', 'lat', 'year','country_id']]

df_grid_per_wiw = df_grid.merge(df_per_wiw, on=['country_id', 'year'])
df_grid_per_wiw = df_grid_per_wiw[['lon', 'lat', 'year','per_wiw']]
df_grid_per_wiw.to_csv(path_part4_grid + 'per_wiw_grid_year.csv', index=False)